In [1]:
from langchain.agents import create_agent
from langchain_google_vertexai import ChatVertexAI
from langgraph.checkpoint.memory import MemorySaver
from toolbox_langchain import ToolboxClient
import os
from dotenv import load_dotenv
import vertexai
load_dotenv()

True

In [2]:
vertexai.init(
    project=os.environ.get("PROJECT_ID"),
    location=os.environ.get("LOCATION")
)

In [7]:
prompt = """
You're a helpful hotel assistant. You handle hotel searching and booking.
When the user searches for a hotel, list the full details for each hotel found: id, name, location, and price tier.
Always use the hotel ID for booking operations.
For any bookings, provide a clear confirmation message.
Don't ask for clarification or confirmation from the user; perform the requested action directly.
"""

In [8]:
async def run_queries(agent_executor):
    config = {"configurable": {"thread_id": "hotel-thread-1"}}

    # --- Query 1: Search for hotels ---
    query1 = "I need to find a hotel in Basel."
    print(f'\n--- USER: "{query1}" ---')
    inputs1 = {"messages": [("user", prompt + query1)]}
    async for event in agent_executor.astream_events(
        inputs1, config=config, version="v2"
    ):
        if event["event"] == "on_chat_model_end" and event["data"]["output"].content:
            print(f"--- AGENT: ---\n{event['data']['output'].content}")

    # --- Query 2: Book a hotel ---
    query2 = "Great, please book the Hyatt Regency Basel for me."
    print(f'\n--- USER: "{query2}" ---')
    inputs2 = {"messages": [("user", query2)]}
    async for event in agent_executor.astream_events(
        inputs2, config=config, version="v2"
    ):
        if event["event"] == "on_chat_model_end" and event["data"]["output"].content:
            print(f"--- AGENT: ---\n{event['data']['output'].content}")    

In [3]:
model = ChatVertexAI(
    model_name="gemini-2.5-flash",
)

/tmp/ipykernel_12602/3116500187.py:1: DeprecationWarning: Use [`ChatGoogleGenerativeAI`][langchain_google_genai.ChatGoogleGenerativeAI] instead.
  model = ChatVertexAI(
/tmp/ipykernel_12602/3116500187.py:1: LangChainDeprecationWarning: The class `ChatVertexAI` was deprecated in LangChain 3.2.0 and will be removed in 4.0.0. An updated version of the class exists in the `langchain-google-genai package and should be used instead. To use it run `pip install -U `langchain-google-genai` and import as `from `langchain_google_genai import ChatGoogleGenerativeAI``.
  model = ChatVertexAI(


In [18]:
async with ToolboxClient("http://127.0.0.1:5000") as client:
    tools = await client.aload_toolset("hotel_toolset")
    agent = create_agent(model, tools, checkpointer=MemorySaver())
    await run_queries(agent)


--- USER: "I need to find a hotel in Basel." ---
--- AGENT: ---
Here are the hotels I found in Basel:

*   **ID:** 1, **Name:** Hilton Basel, **Location:** Basel, **Price Tier:** Luxury
*   **ID:** 3, **Name:** Hyatt Regency Basel, **Location:** Basel, **Price Tier:** Upper Upscale

--- USER: "Great, please book the Hyatt Regency Basel for me." ---
--- AGENT: ---
Your booking for Hyatt Regency Basel is confirmed.
